# Tenacious-Bench Week 11 — STABLE DPO Training (No Unsloth)
**Model:** Qwen2.5-1.5B-Instruct (4-bit QLoRA)
**Method:** Direct Preference Optimization (DPO)
**Environment:** Google Colab T4 GPU

This notebook bypasses the buggy Unsloth libraries and uses the standard, industry-default HuggingFace tools (`transformers`, `peft`, `trl`). This ensures the code is **stable** and won't crash on import.

## Step 1 — Install Stable Dependencies
(Wait for this to finish, then restart the session if prompted)

In [ ]:
!pip install -U pip
!pip install -U transformers peft trl bitsandbytes accelerate datasets matplotlib
print('✅ Standard libraries installed!')

## Step 2 — Mount Google Drive
(To save your model permanently)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted!')

## Step 3 — Setup Environment and Load Model
**Note:** Qwen2.5-1.5B fits comfortably in 4-bit on a T4 GPU.

In [ ]:
import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import DPOTrainer, DPOConfig
from datasets import Dataset, disable_caching

# 1. Setup paths
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
output_dir = "/content/drive/MyDrive/Tenacious-DPO-Final"

# 2. Load 4-bit Quantized Model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# 3. Configure LoRA
peft_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
print(f"✅ Model loaded. Trainable parameters: {model.get_nb_trainable_parameters()}")

## Step 4 — Load and Format Datasets
**Important:** Upload your `preferences_train.jsonl` and `preferences_dev.jsonl` to the Colab file side-panel before running this cell.

In [ ]:
# Disable caching to prevent OOM pickling issues in Colab
disable_caching()

def read_jsonl(path):
    with open(path) as f: return [json.loads(line) for line in f if line.strip()]

def format_row(row):
    return {
        "prompt": f"<|im_start|>user\n{row['prompt']}<|im_end|>\n<|im_start|>assistant\n",
        "chosen": f"{row['chosen']}<|im_end|>",
        "rejected": f"{row['rejected']}<|im_end|>"
    }

train_ds = Dataset.from_list([format_row(r) for r in read_jsonl("/content/preferences_train.jsonl")])
eval_ds = Dataset.from_list([format_row(r) for r in read_jsonl("/content/preferences_dev.jsonl")])

print(f"✅ Loaded {len(train_ds)} training pairs and {len(eval_ds)} dev pairs.")
print("Sample Formatted Prompt:")
print(train_ds[0]['prompt'])

## Step 5 — Launch DPO Training
**Protocol:** LR=1e-4, beta=0.1, 60 steps.

In [ ]:
trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=DPOConfig(
        output_dir=output_dir,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=1e-4,        # Official Week 11 Methodology
        beta=0.1,                  # Official Week 11 Methodology
        max_steps=60,
        logging_steps=1,
        save_steps=60,
        fp16=True,
        remove_unused_columns=False,
        report_to="none"
    ),
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    max_length=1024, max_prompt_length=512,
)

print("🚀 Starting Training (Watch for Chosen vs Rejected rewards)... ")
trainer.train()
print("🏁 Training Complete!")

## Step 6 — Visualize Rewards & Convergence
This shows you exactly how well the model learned to prefer the 'Chosen' responses over the 'Rejected' ones.

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
steps = [x['step'] for x in history if 'loss' in x]
loss = [x['loss'] for x in history if 'loss' in x]
c_rewards = [x['rewards/chosen'] for x in history if 'rewards/chosen' in x]
r_rewards = [x['rewards/rejected'] for x in history if 'rewards/rejected' in x]
margins = [x['rewards/margins'] for x in history if 'rewards/margins' in x]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Loss
ax1.plot(steps, loss, label='Training Loss', color='gray', alpha=0.5)
ax1.set_title('DPO Training Loss')
ax1.set_xlabel('Step')
ax1.legend()

# Plot 2: Rewards & Margins
ax2.plot(steps, c_rewards, label='Chosen Rewards', color='green')
ax2.plot(steps, r_rewards, label='Rejected Rewards', color='red')
ax2.plot(steps, margins, label='Margin', color='blue', linestyle='--')
ax2.set_title('DPO Rewards & Preference Margin')
ax2.set_xlabel('Step')
ax2.legend()

plt.tight_layout()
plt.show()

## Step 7 — Quick Inference Test
Verify the model's behavior on a sample sales objection.

In [ ]:
prompt = """<|im_start|>user
I'm concerned about the series G funding news. Is your engineering bench stable enough to support our 6-month roadmap?<|im_end|>
<|im_start|>assistant
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.7)

print("--- Model Output ---")
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

## Step 8 — Push to HuggingFace

In [ ]:
from huggingface_hub import HfApi
HF_TOKEN = "hf_YOUR_WRITE_TOKEN_HERE"   # <-- PASTE YOUR WRITE TOKEN
api = HfApi()
user = api.whoami(token=HF_TOKEN)['name']

api.upload_folder(
    folder_path=f"{output_dir}/checkpoint-60",
    repo_id=f"{user}/Tenacious-Qwen-DPO-Stable",
    repo_type="model",
    token=HF_TOKEN
)
print(f"✅ Model pushed to: https://huggingface.co/{user}/Tenacious-Qwen-DPO-Stable")